### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

#### 1️⃣ 필요한 라이브러리 설치 및 불러오기

In [ ]:
# Hugging Face transformers 및 관련 라이브러리 설치
!pip install transformers pillow torch torchvision accelerate -q

In [1]:
import os
from tqdm import tqdm
import torch
from PIL import Image
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from pathlib import Path

print("✅ 라이브러리 로드 완료")

/home/nute11a/anaconda3/envs/SAT/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 라이브러리 로드 완료


---
#### 2️⃣ 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/06_car_damage_classification/code"

In [3]:
# Google Drive 내 작업 디렉토리 경로 설정
WORK_DIR = '/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/06_car_damage_classification'
WORK_DIR = '/home/nute11a/workspace/2026_AI_Advanced_Study/4차시/06_car_damage_classification'
# 작업 디렉토리로 이동
os.chdir(WORK_DIR)
print(f"현재 작업 디렉토리: {WORK_DIR}")

# 데이터 경로 확인
DATA_DIR = Path(f'{WORK_DIR}/data')
if DATA_DIR.exists():
    print(f"✅ 데이터 디렉토리 발견: {DATA_DIR}")
else:
    print(f"❌ 데이터 디렉토리를 찾을 수 없습니다: {DATA_DIR}")

현재 작업 디렉토리: /home/nute11a/workspace/2026_AI_Advanced_Study/4차시/06_car_damage_classification
✅ 데이터 디렉토리 발견: /home/nute11a/workspace/2026_AI_Advanced_Study/4차시/06_car_damage_classification/data


---

#### 3️⃣ 작업 디렉토리 및 모델 경로 설정

중요: `WORK_DIR`을 본인의 Google Drive 경로로 수정하세요.

In [4]:
# Google Drive 내 작업 디렉토리 경로 설정
# 학습된 모델 경로 (고정)
MODEL_PATH = os.path.join(WORK_DIR, 'runs/classification/final_model')

# 작업 디렉토리로 이동
os.chdir(WORK_DIR)
print(f"현재 작업 디렉토리: {os.getcwd()}")

# 모델 파일 존재 확인
if os.path.exists(MODEL_PATH):
    print(f"✅ 모델 발견: {MODEL_PATH}")
else:
    print(f"❌ 모델을 찾을 수 없습니다: {MODEL_PATH}")
    print("   먼저 01_train.ipynb를 실행하여 모델을 학습하세요.")

현재 작업 디렉토리: /home/nute11a/workspace/2026_AI_Advanced_Study/4차시/06_car_damage_classification
✅ 모델 발견: /home/nute11a/workspace/2026_AI_Advanced_Study/4차시/06_car_damage_classification/runs/classification/final_model


---
#### 4️⃣ 커스텀 데이터셋 클래스 정의


In [5]:
class CarDmgDataset(TorchDataset):
    """로컬 이미지 파일을 로드하는 커스텀 데이터셋"""
    
    def __init__(self, data_dir, processor=None):
        self.data_dir = Path(data_dir)
        self.processor = processor
        self.samples = []
        
        # 클래스별 이미지 수집
        for cls_info in os.listdir(self.data_dir):
            cls_idx, cls_name = cls_info.split('_',1)
            cls_idx = int(cls_idx)
            for img_path in sorted((self.data_dir / cls_info).glob('*.jpg')):
                self.samples.append((img_path, cls_idx))
            
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        return {'image': image, 'label': label}

print("✅ 커스텀 데이터셋 클래스 정의 완료")

✅ 커스텀 데이터셋 클래스 정의 완료


---
#### 5️⃣ 모델 및 이미지 프로세서 로드

In [ ]:
print("🤖 모델 로드 중...")

# 🎯 [미션] 저장된 경로(MODEL_PATH)에서 학습된 모델을 불러오는 함수를 완성하세요
model = AutoModelForImageClassification._______(MODEL_PATH)
processor = AutoImageProcessor.from_pretrained(MODEL_PATH)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"✅ 모델 로드 완료 (Device: {device})")

🤖 모델 로드 중...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ 모델 로드 완료 (Device: cuda)


---
#### 6️⃣ 데이터셋 로드

In [ ]:
print("📥 로컬 데이터셋 로드 중...\n")

# 🎯 [미션] 'data' 폴더 안에 있는 '테스트용' 데이터 폴더 이름을 적으세요.
test_dataset = CarDmgDataset('____', processor=processor)

print(f"✅ 테스트 데이터 준비 완료: {len(test_dataset)}장")

📥 로컬 데이터셋 로드 중...

✅ 테스트 데이터 준비 완료: 116장


---
#### 7️⃣ 모델 평가 수행

In [ ]:

print("🔍 평가 시작...\n")

all_predictions = []
all_labels = []

# 🎯 [미션]  평가 중에는 모델이 학습(기록)을 하지 않도록 막아주는 기능을 사용합니다.
with torch.no_grad():
    for item in tqdm(test_dataset):
        image = item['image']
        label = item['label']
        
        # 전처리
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 예측
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
        
        all_predictions.append(predictions.cpu().item())
        all_labels.append(label)

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

print("✅ 평가 완료!")

🔍 평가 시작...



100%|██████████| 116/116 [00:03<00:00, 29.22it/s]

✅ 평가 완료!


---
#### 8️⃣ 성능 지표 확인

In [2]:
# 전체 성능
accuracy = accuracy_score(all_labels, all_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_predictions, average='weighted', zero_division=0
)

print("\n" + "="*50)
print("=== 전체 성능 (Overall Performance) ===")
print("="*50)
print(f"\n📊 Accuracy: {accuracy:.4f}")
print(f"📊 Precision: {precision:.4f}")
print(f"📊 Recall: {recall:.4f}")
print(f"📊 F1-Score: {f1:.4f}")

# 클래스별 성능
print("\n" + "="*50)
print("=== 클래스별 성능 (Per-Class Performance) ===")
print("="*50)

precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
    all_labels, all_predictions, average=None, zero_division=0
)

for i in range(5):
    print(f"\n📌 Class {i}:")
    print(f"precision | recall | f1-score")
    print(f"{precision_per_class[i]:.4f}    | {recall_per_class[i]:.4f} | {f1_per_class[i]:.4f}")

NameError: name 'all_labels' is not defined